# 9.1.门控循环单元（GRU）

在 [8.7节](../08_pypto_recurrent_neural_networks/08.07_bptt.ipynb)中， 我们讨论了如何在循环神经网络中计算梯度， 以及矩阵连续乘积可以导致梯度消失或梯度爆炸的问题。 下面我们简单思考一下这种梯度异常在实践中的意义：

* 我们可能会遇到这样的情况：早期观测值对预测所有未来观测值具有非常重要的意义。考虑一个极端情况，其中第一个观测值包含一个校验和，目标是在序列的末尾辨别校验和是否正确。在这种情况下，第一个词元的影响至关重要。我们希望有某些机制能够在一个记忆元里存储重要的早期信息。如果没有这样的机制，我们将不得不给这个观测值指定一个非常大的梯度，因为它会影响所有后续的观测值。
* 我们可能会遇到这样的情况：一些词元没有相关的观测值。例如，在对网页内容进行情感分析时，可能有一些辅助HTML代码与网页传达的情绪无关。我们希望有一些机制来*跳过*隐状态表示中的此类词元。
* 我们可能会遇到这样的情况：序列的各个部分之间存在逻辑中断。例如，书的章节之间可能会有过渡存在，或者证券的熊市和牛市之间可能会有过渡存在。在这种情况下，最好有一种方法来*重置*我们的内部状态表示。

在学术界已经提出了许多方法来解决这类问题。 其中最早的方法是"长短期记忆"（long-short-term memory，LSTM） ([Hochreiter, 1997](https://zh.d2l.ai/chapter_references/zreferences.html#Hochreiter.Schmidhuber.1997))， 我们将在 [9.2节](09.02_lstm.ipynb)中讨论。 门控循环单元（gated recurrent unit，GRU） ([Cho et al., 2014](https://zh.d2l.ai/chapter_references/zreferences.html#Cho.Van-Merrienboer.Bahdanau.ea.2014)) 是一个稍微简化的变体，通常能够提供同等的效果， 并且计算 ([Chung et al., 2014](https://zh.d2l.ai/chapter_references/zreferences.html#Chung.Gulcehre.Cho.ea.2014))的速度明显更快。 由于门控循环单元更简单，我们从它开始解读。



---
## 9.1.1.环境配置

In [1]:
%pip install pypto==0.2.0 torch torch_npu matplotlib

In [2]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import pypto
import torch
from torch import nn
from torch.nn import functional as F
import torch_npu
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

In [3]:
from src.utils import load_data_time_machine

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
import torch
from torch import nn
from d2l import torch as d2l


batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)
    </pre>
  </div>
</details>


---
## 9.1.2.门控隐状态

门控循环单元与普通的循环神经网络之间的关键区别在于： 前者支持隐状态的门控。 这意味着模型有专门的机制来确定应该何时更新隐状态， 以及应该何时重置隐状态。 这些机制是可学习的，并且能够解决上面列出的问题。 例如，如果第一个词元非常重要， 模型将学会在第一次观测之后不更新隐状态。 同样，模型也可以学会跳过不相关的临时观测。 最后，模型还将学会在需要的时候重置隐状态。 下面我们将详细讨论各类门控。

### 9.1.2.1.重置门和更新门

我们首先介绍*重置门*（reset gate）和*更新门*（update gate）。 我们把它们设计成$(0, 1)$区间中的向量， 这样我们就可以进行凸组合。 重置门允许我们控制"可能还想记住"的过去状态的数量； 更新门将允许我们控制新状态中有多少个是旧状态的副本。

我们从构造这些门控开始。下图描述了门控循环单元中的重置门和更新门的输入， 输入是由当前时间步的输入和前一时间步的隐状态给出。 两个门的输出是由使用sigmoid激活函数的两个全连接层给出。

<div align="center">
  <img src="./images/gru-1.svg" alt="图9.1.1 在门控循环单元模型中计算重置门和更新门" width="400">
  <br><small>图9.1.1 在门控循环单元模型中计算重置门和更新门</small>
</div>

我们来看一下门控循环单元的数学表达。 对于给定的时间步$t$，假设输入是一个小批量 $\mathbf{X}_t \in \mathbb{R}^{n \times d}$ （样本个数$n$，输入个数$d$）， 上一个时间步的隐状态是 $\mathbf{H}_{t-1} \in \mathbb{R}^{n \times h}$ （隐藏单元个数$h$）。 那么，重置门$\mathbf{R}_t \in \mathbb{R}^{n \times h}$和 更新门$\mathbf{Z}_t \in \mathbb{R}^{n \times h}$的计算如下所示：

$$
\tag{9.1.1}
\begin{aligned}
\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xr} + \mathbf{H}_{t-1} \mathbf{W}_{hr} + \mathbf{b}_r),\\
\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xz} + \mathbf{H}_{t-1} \mathbf{W}_{hz} + \mathbf{b}_z),
\end{aligned}
$$

其中$\mathbf{W}_{xr}, \mathbf{W}_{xz} \in \mathbb{R}^{d \times h}$ 和$\mathbf{W}_{hr}, \mathbf{W}_{hz} \in \mathbb{R}^{h \times h}$是权重参数， $\mathbf{b}_r, \mathbf{b}_z \in \mathbb{R}^{1 \times h}$是偏置参数。 请注意，在求和过程中会触发广播机制 （请参阅 [2.1节](../02_pypto_preliminaries/02.02_data_manipulation.ipynb)）。 我们使用sigmoid函数（如 [4.1节](../04_pypto_multilayer_perceptrons/04.01_mlp.ipynb)中介绍的） 将输入值转换到区间$(0, 1)$。

### 9.1.2.2.候选隐状态

接下来，让我们将重置门$\mathbf{R}_t$ 与公式(8.4.5)中的常规隐状态更新机制集成， 得到在时间步$t$的*候选隐状态*（candidate hidden state） $\tilde{\mathbf{H}}_t \in \mathbb{R}^{n \times h}$。

$$\tag{9.1.2}\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{xh} + \left(\mathbf{R}_t \odot \mathbf{H}_{t-1}\right) \mathbf{W}_{hh} + \mathbf{b}_h),$$

其中$\mathbf{W}_{xh} \in \mathbb{R}^{d \times h}$ 和$\mathbf{W}_{hh} \in \mathbb{R}^{h \times h}$是权重参数， $\mathbf{b}_h \in \mathbb{R}^{1 \times h}$是偏置项， 符号$\odot$是Hadamard积（按元素乘积）运算符。 在这里，我们使用tanh非线性激活函数来确保候选隐状态中的值保持在区间$(-1, 1)$中。

与公式(8.4.5)相比，候选隐状态中的$\mathbf{R}_t$和$\mathbf{H}_{t-1}$ 的元素相乘可以减少以往状态的影响。 每当重置门$\mathbf{R}_t$中的项接近$1$时， 我们恢复一个如公式(8.4.5)中的普通的循环神经网络。 对于重置门$\mathbf{R}_t$中所有接近$0$的项， 候选隐状态是以$\mathbf{X}_t$作为输入的多层感知机的结果。 因此，任何预先存在的隐状态都会被*重置*为默认值。下图说明了应用重置门之后的计算流程。

<div align="center">
  <img src="./images/gru-2.svg" alt="图9.1.2 在门控循环单元模型中计算候选隐状态" width="400">
  <br><small>图9.1.2 在门控循环单元模型中计算候选隐状态</small>
</div>

### 9.1.2.3.隐状态

上述的计算结果只是候选隐状态，我们仍然需要结合更新门$\mathbf{Z}_t$的效果。 这一步确定新的隐状态$\mathbf{H}_t \in \mathbb{R}^{n \times h}$ 在多大程度上来自旧的状态$\mathbf{H}_{t-1}$和 新的候选状态$\tilde{\mathbf{H}}_t$。 更新门$\mathbf{Z}_t$仅需要在 $\mathbf{H}_{t-1}$和$\tilde{\mathbf{H}}_t$ 之间进行按元素的凸组合就可以实现这个目标。 这就得出了门控循环单元的最终更新公式：

$$\tag{9.1.3}\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1} + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t.$$

每当更新门$\mathbf{Z}_t$接近$1$时，模型就倾向只保留旧状态。 此时，来自$\mathbf{X}_t$的信息基本上被忽略， 从而有效地跳过了依赖链条中的时间步$t$。 相反，当$\mathbf{Z}_t$接近$0$时， 新的隐状态$\mathbf{H}_t$就会接近候选隐状态$\tilde{\mathbf{H}}_t$。 这些设计可以帮助我们处理循环神经网络中的梯度消失问题， 并更好地捕获时间步距离很长的序列的依赖关系。 例如，如果整个子序列的所有时间步的更新门都接近于$1$， 则无论序列的长度如何，在序列起始时间步的旧隐状态都将很容易保留并传递到序列结束。下图说明了更新门起作用后的计算流程。

<div align="center">
  <img src="./images/gru-3.svg" alt="图9.1.3 计算门控循环单元模型中的隐状态" width="400">
  <br><small>图9.1.3 计算门控循环单元模型中的隐状态</small>
</div>

总之，门控循环单元具有以下两个显著特征：

* 重置门有助于捕获序列中的短期依赖关系；
* 更新门有助于捕获序列中的长期依赖关系。

---
## 9.1.3.从零开始实现

为了更好地理解门控循环单元模型，我们从零开始实现它。 首先，我们读取 [8.5节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)中使用的时间机器数据集：

### 9.1.3.1.初始化模型参数

下一步是初始化模型参数。 我们从标准差为$0.01$的高斯分布中提取权重， 并将偏置项设为$0$，超参数`num_hiddens`定义隐藏单元的数量， 实例化与更新门、重置门、候选隐状态和输出层相关的所有权重和偏置。

In [4]:
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size

    def normal(shape):
        return torch.randn(size=shape, device=device)*0.01

    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))

    W_xz, W_hz, b_z = three()  # 更新门参数
    W_xr, W_hr, b_r = three()  # 重置门参数
    W_xh, W_hh, b_h = three()  # 候选隐状态参数
    # 输出层参数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    # 附加梯度
    params = [W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device)*0.01
    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    W_xz, W_hz, b_z = three()  # 更新门参数
    W_xr, W_hr, b_r = three()  # 重置门参数
    W_xh, W_hh, b_h = three()  # 候选隐状态参数
    # 输出层参数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    # 附加梯度
    params = [W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params
    </pre>
  </div>
</details>


### 9.1.3.2.定义模型

现在我们将**定义隐状态的初始化函数**`init_gru_state`。 与 [8.5节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)中定义的`init_rnn_state`函数一样， 此函数返回一个形状为（批量大小，隐藏单元个数）的张量，张量的值全部为零。

In [5]:
def init_gru_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device), )

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def init_gru_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device), )
    </pre>
  </div>
</details>


---
## 9.1.4.GRU 专用算子：sigmoid、mul 和 sub

matmul、bias_add 算子已在 [4.2 节](../04_pypto_multilayer_perceptrons/04.02_mlp_scratch.ipynb)实现，add、tanh 算子已在 [8.5 节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)实现——从 `src.pypto_ops` 直接导入即可复用。GRU 公式中还需要三个新算子：**sigmoid**（门控激活）、**mul**（逐元素乘法/Hadamard 积）和 **sub**（逐元素减法，用于 $1-Z$）。

为保持教学完整性，**本节将内联展示这三个算子的完整 Kernel + 工厂函数链条**，帮助读者看清 GRU 在 NPU 上的完整工作流。后续[9.2 节](./09.02_lstm.ipynb)将直接从 `src/pypto_ops` 导入，不再重复。


In [6]:
from src.pypto_ops import PyPTOMatmul, PyPTOBiasAdd, PyPTOAdd, PyPTOTanh, loss_fn

# ── sigmoid：y = σ(x) = 1/(1+e^(-x))，反向：grad_x = y·(1-y)·grad_y ──
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def sigmoid_fwd_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    b.move(pypto.sigmoid(a))

@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def sigmoid_bwd_kernel(
    b: pypto.Tensor([], pypto.DT_FP32),
    grad_b: pypto.Tensor([], pypto.DT_FP32),
    grad_a: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    one_minus_b = pypto.neg(pypto.sub(b, 1.0))
    g_mul_b = pypto.mul(grad_b, b)
    grad_a.move(pypto.mul(g_mul_b, one_minus_b))

def make_pypto_sigmoid(fwd_kernel, bwd_kernel):
    class PyPTOSigmoidImpl(torch.autograd.Function):
        @staticmethod
        def forward(ctx, a):
            b = torch.empty_like(a)
            fwd_kernel(a, b)
            ctx.save_for_backward(b)
            return b
        @staticmethod
        def backward(ctx, grad_b):
            (b,) = ctx.saved_tensors
            grad_a = torch.empty_like(b) if ctx.needs_input_grad[0] else None
            if grad_a is not None:
                bwd_kernel(b.contiguous(), grad_b.contiguous(), grad_a)
            return grad_a
    return PyPTOSigmoidImpl

# ── mul：c = a ⊙ b，反向：grad_a = grad_c ⊙ b, grad_b = grad_c ⊙ a ──
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def mul_fwd_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    c: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    c.move(pypto.mul(a, b))

@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def mul_bwd_kernel(
    grad_c: pypto.Tensor([], pypto.DT_FP32),
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    grad_a: pypto.Tensor([], pypto.DT_FP32),
    grad_b: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    grad_a.move(pypto.mul(grad_c, b))
    grad_b.move(pypto.mul(grad_c, a))

def make_pypto_mul(fwd_kernel, bwd_kernel):
    class PyPTOMulImpl(torch.autograd.Function):
        @staticmethod
        def forward(ctx, a, b):
            ctx.save_for_backward(a, b)
            c = torch.empty_like(a)
            fwd_kernel(a, b, c)
            return c
        @staticmethod
        def backward(ctx, grad_c):
            a, b = ctx.saved_tensors
            need_grad_a = ctx.needs_input_grad[0]
            need_grad_b = ctx.needs_input_grad[1]
            grad_a = torch.empty_like(a) if need_grad_a else None
            grad_b = torch.empty_like(b) if need_grad_b else None
            tmp_a = grad_a if need_grad_a else torch.empty_like(a)
            tmp_b = grad_b if need_grad_b else torch.empty_like(b)
            if need_grad_a or need_grad_b:
                bwd_kernel(grad_c.contiguous(), a.contiguous(),
                           b.contiguous(), tmp_a, tmp_b)
            return grad_a, grad_b
    return PyPTOMulImpl

# ── sub：c = a - b，反向：grad_a = grad_c, grad_b = -grad_c ──
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def sub_fwd_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    c: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    c.move(pypto.sub(a, b))

@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def sub_bwd_grad_b_kernel(
    grad_c: pypto.Tensor([], pypto.DT_FP32),
    grad_b: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    grad_b.move(pypto.neg(grad_c))

def make_pypto_sub(fwd_kernel, bwd_grad_b_kernel):
    class PyPTOSubImpl(torch.autograd.Function):
        @staticmethod
        def forward(ctx, a, b):
            ctx.save_for_backward(a, b)
            c = torch.empty_like(a)
            fwd_kernel(a, b, c)
            return c
        @staticmethod
        def backward(ctx, grad_c):
            need_grad_a = ctx.needs_input_grad[0]
            need_grad_b = ctx.needs_input_grad[1]
            grad_a = grad_c if need_grad_a else None
            grad_b = torch.empty_like(grad_c) if need_grad_b else None
            if need_grad_b:
                bwd_grad_b_kernel(grad_c.contiguous(), grad_b)
            return grad_a, grad_b
    return PyPTOSubImpl

PyPTOSigmoid = make_pypto_sigmoid(sigmoid_fwd_kernel, sigmoid_bwd_kernel)
PyPTOMul = make_pypto_mul(mul_fwd_kernel, mul_bwd_kernel)
PyPTOSub = make_pypto_sub(sub_fwd_kernel, sub_bwd_grad_b_kernel)

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><b>sigmoid 算子</b>：前向使用 <code>pypto.sigmoid(a)</code> 原生 API；反向利用恒等式 <code>dσ/dx = σ(x)·(1-σ(x))</code>，前向输出 <code>b</code>（即 σ 值）直接作为反向输入，无需重算。</li>
      <li style="margin: 0 0 8px 0;"><b>mul 算子</b>：前向执行 <code>pypto.mul(a, b)</code> 逐元素 Hadamard 积；反向调用双边梯度核 <code>mul_bwd_kernel</code> 在同一次核调用中计算 <code>grad_a = grad_c ⊙ b</code> 和 <code>grad_b = grad_c ⊙ a</code>。</li>
      <li style="margin: 0 0 8px 0;"><b>sub 算子</b>：前向执行 <code>pypto.sub(a, b)</code> 逐元素减法；反向：<code>grad_a = grad_c</code>（恒等映射，直接复用上游梯度），<code>grad_b = -grad_c</code>（调用独立的 <code>sub_bwd_grad_b_kernel</code> 做 <code>pypto.neg</code>）。</li>
      <li style="margin: 0 0 8px 0;">这三个算子已收录在 <code>src/pypto_ops.py</code> 中，<b>后续章节可直接导入</b>（见 <a href="./09.02_lstm.ipynb">9.2 节</a>）。</li>
    </ul>
  </div>
</details>


现在我们准备**定义门控循环单元模型**， 模型的架构与基本的循环神经网络单元是相同的， 只是权重更新公式更为复杂。

In [7]:
def gru_pypto(inputs, state, params):
    """使用 PyPTO 算子执行 GRU 前向传播。"""
    W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        # ── 更新门 Z = σ(X·W_xz + H·W_hz + b_z) ──
        XW_z = PyPTOMatmul.apply(X, W_xz)
        HW_z = PyPTOMatmul.apply(H, W_hz)
        z_sum = PyPTOAdd.apply(XW_z, HW_z)
        z_biased = PyPTOBiasAdd.apply(z_sum, b_z)
        Z = PyPTOSigmoid.apply(z_biased)

        # ── 重置门 R = σ(X·W_xr + H·W_hr + b_r) ──
        XW_r = PyPTOMatmul.apply(X, W_xr)
        HW_r = PyPTOMatmul.apply(H, W_hr)
        r_sum = PyPTOAdd.apply(XW_r, HW_r)
        r_biased = PyPTOBiasAdd.apply(r_sum, b_r)
        R = PyPTOSigmoid.apply(r_biased)

        # ── 候选隐状态 H̃ = tanh(X·W_xh + (R⊙H)·W_hh + b_h) ──
        XW_h = PyPTOMatmul.apply(X, W_xh)
        RH = PyPTOMul.apply(R, H)
        RHW = PyPTOMatmul.apply(RH, W_hh)
        h_sum = PyPTOAdd.apply(XW_h, RHW)
        h_biased = PyPTOBiasAdd.apply(h_sum, b_h)
        H_tilde = PyPTOTanh.apply(h_biased)

        # ── 隐状态更新 H = Z⊙H + (1-Z)⊙H̃ ──
        ZH = PyPTOMul.apply(Z, H)
        ones = torch.ones_like(Z)
        one_minus_z = PyPTOSub.apply(ones, Z)
        inv_z_ht = PyPTOMul.apply(one_minus_z, H_tilde)
        H = PyPTOAdd.apply(ZH, inv_z_ht)

        # ── 输出 Y = H·W_hq + b_q ──
        Y = PyPTOBiasAdd.apply(PyPTOMatmul.apply(H, W_hq), b_q)
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def gru(inputs, state, params):
    W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        Z = torch.sigmoid((X @ W_xz) + (H @ W_hz) + b_z)
        R = torch.sigmoid((X @ W_xr) + (H @ W_hr) + b_r)
        H_tilde = torch.tanh((X @ W_xh) + ((R * H) @ W_hh) + b_h)
        H = Z * H + (1 - Z) * H_tilde
        Y = H @ W_hq + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)
    </pre>
  </div>
</details>


接下来，我们**定义训练用的模型类**，封装参数初始化、状态初始化和前向传播：


In [8]:
class PyPTORNNModelScratch:
    """PyPTO 从零实现的 RNN/GRU 模型封装。"""
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params_fn=None, init_state_fn=None, forward_fn=None):
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.device = device
        self.params = (get_params_fn or get_params)(vocab_size, num_hiddens, device)
        self.init_state = init_state_fn or init_gru_state
        self.forward_fn = forward_fn or gru_pypto

    def __call__(self, X, state):
        X = torch.nn.functional.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)

    def begin_state(self, batch_size, device=None):
        if device is None:
            device = self.device
        return self.init_state(batch_size, self.num_hiddens, device)


In [9]:
# 形状检查：验证 GRU 模型的输入/输出维度
num_hiddens = 512
net = PyPTORNNModelScratch(len(vocab), num_hiddens, device, get_params,
                           init_gru_state, gru_pypto)
X = torch.arange(10).reshape((2, 5)).to(device)
state = net.begin_state(X.shape[0], device)
Y, new_state = net(X, state)
Y.shape, len(new_state), new_state[0].shape


(torch.Size([10, 28]), 1, torch.Size([2, 512]))

输出中 `10 = 2 × 5`（批量大小 × 时间步），`28` 为词表大小；状态元组含 1 项，形状为 `(batch=2, num_hiddens=512)`，即 GRU 只需一个隐状态。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class RNNModelScratch:
    """从零开始实现的循环神经网络模型"""
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn
    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)
    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)
    </pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_hiddens = 512
net = RNNModelScratch(len(vocab), num_hiddens, d2l.try_gpu(), get_params,
                      init_gru_state, gru)
X = torch.arange(10).reshape((2, 5))
state = net.begin_state(X.shape[0], d2l.try_gpu())
Y, new_state = net(X.to(d2l.try_gpu()), state)
Y.shape, len(new_state), new_state[0].shape
    </pre>
  </div>
</details>


---
## 9.1.5.预测

让我们**首先定义预测函数来生成`prefix`之后的新字符**，其中的`prefix`是一个用户提供的包含多个字符的字符串。在循环遍历`prefix`中的开始字符时，我们不断地将隐状态传递到下一个时间步，但是不生成任何输出。这被称为*预热*（warm-up）期，因为在此期间模型会自我更新（例如，更新隐状态），但不会进行预测。预热期结束后，隐状态的值通常比刚开始的初始值更适合预测，从而预测字符并输出它们。

In [10]:
def predict_ch8(prefix, num_preds, net, vocab, device):
    """在 prefix 后生成新字符。"""
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]: # 预热期
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds): # 预测num_preds步
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return "".join([vocab.idx_to_token[i] for i in outputs])

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def predict_ch8(prefix, num_preds, net, vocab, device):
    """在prefix后面生成新字符"""
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:  # 预热期
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):  # 预测num_preds步
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return "".join([vocab.idx_to_token[i] for i in outputs])
    </pre>
  </div>
</details>


---
## 9.1.6.梯度裁剪

对于长度为$T$的序列，我们在迭代中计算这$T$个时间步上的梯度，将会在反向传播过程中产生长度为$\mathcal{O}(T)$的矩阵乘法链。如 [4.8节](../04_pypto_multilayer_perceptrons/04.08_numerical_stability_and_init.ipynb)所述，当$T$较大时，它可能导致数值不稳定，例如可能导致梯度爆炸或梯度消失。因此，循环神经网络模型往往需要额外的方式来支持稳定训练。

一般来说，当解决优化问题时，我们对模型参数采用更新步骤。假定在向量形式的$\mathbf{x}$中，或者在小批量数据的负梯度$\mathbf{g}$方向上。例如，使用$\eta > 0$作为学习率时，在一次迭代中，我们将$\mathbf{x}$更新为$\mathbf{x} - \eta \mathbf{g}$。如果我们进一步假设目标函数$f$表现良好，即函数$f$在常数$L$下是*利普希茨连续的*（Lipschitz continuous）。也就是说，对于任意$\mathbf{x}$和$\mathbf{y}$我们有：

$$\tag{9.1.4}|f(\mathbf{x}) - f(\mathbf{y})| \leq L \|\mathbf{x} - \mathbf{y}\|.$$

在这种情况下，我们可以安全地假设：如果我们通过$\eta \mathbf{g}$更新参数向量，则

$$\tag{9.1.5}|f(\mathbf{x}) - f(\mathbf{x} - \eta\mathbf{g})| \leq L \eta\|\mathbf{g}\|,$$

这意味着我们不会观察到超过$L \eta \|\mathbf{g}\|$的变化。这既是坏事也是好事。坏的方面，它限制了取得进展的速度；好的方面，它限制了事情变糟的程度，尤其当我们朝着错误的方向前进时。

有时梯度可能很大，从而优化算法可能无法收敛。我们可以通过降低$\eta$的学习率来解决这个问题。但是如果我们很少得到大的梯度呢？在这种情况下，这种做法似乎毫无道理。一个流行的替代方案是通过将梯度$\mathbf{g}$投影回给定半径（例如$\theta$）的球来裁剪梯度$\mathbf{g}$。如下式：

$$\tag{9.1.6}\mathbf{g} \leftarrow \min\left(1, \frac{\theta}{\|\mathbf{g}\|}\right) \mathbf{g}.$$

通过这样做，我们知道梯度范数永远不会超过$\theta$，并且更新后的梯度完全与$\mathbf{g}$的原始方向对齐。它还有一个值得拥有的副作用，即限制任何给定的小批量数据（以及其中任何给定的样本）对参数向量的影响，这赋予了模型一定程度的稳定性。梯度裁剪提供了一个快速修复梯度爆炸的方法，虽然它并不能完全解决问题，但它是众多有效的技术之一。

下面我们定义一个函数来裁剪模型的梯度，模型是从零开始实现的模型或由高级API构建的模型。我们在此计算了所有模型参数的梯度的范数。

In [11]:
def grad_clipping(net, theta):
    """裁剪梯度。"""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def grad_clipping(net, theta):
    """裁剪梯度"""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm
    </pre>
  </div>
</details>


---
## 9.1.7.训练

在训练模型之前，让我们**定义一个函数在一个迭代周期内训练模型**。它与我们训练 [3.6节](../03_pypto_linear_networks/03.06_softmax_regression_scratch.ipynb)模型的方式有三个不同之处。

1. 序列数据的不同采样方法（随机采样和顺序分区）将导致隐状态初始化的差异。
1. 我们在更新模型参数之前裁剪梯度。
   这样的操作的目的是，即使训练过程中某个点上发生了梯度爆炸，也能保证模型不会发散。
1. 我们用困惑度来评价模型。如 8.4.4节所述，
   这样的度量确保了不同长度的序列具有可比性。

具体来说，当使用顺序分区时，我们只在每个迭代周期的开始位置初始化隐状态。由于下一个小批量数据中的第$i$个子序列样本与当前第$i$个子序列样本相邻，因此当前小批量数据最后一个样本的隐状态，将用于初始化下一个小批量数据第一个样本的隐状态。这样，存储在隐状态中的序列的历史信息可以在一个迭代周期内流经相邻的子序列。然而，在任何一点隐状态的计算，都依赖于同一迭代周期中前面所有的小批量数据，这使得梯度计算变得复杂。为了降低计算量，在处理任何一个小批量数据之前，我们先分离梯度，使得隐状态的梯度计算总是限制在一个小批量数据的时间步内。

当使用随机抽样时，因为每个样本都是在一个随机位置抽样的，因此需要为每个迭代周期重新初始化隐状态。与 [3.6节](../03_pypto_linear_networks/03.06_softmax_regression_scratch.ipynb)中的`train_epoch_ch3`函数相同，`updater`是更新模型参数的常用函数。它既可以是从头开始实现的`d2l.sgd`函数，也可以是深度学习框架中内置的优化函数。


In [12]:
import math
from src.utils import Timer, Accumulator, sgd

def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    """训练网络一个迭代周期"""
    state, timer = None, Timer()
    metric = Accumulator(2)
    for X, Y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state = state.detach()
            else:
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long())
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric.add(l * y.numel(), y.numel())
    return math.exp(metric[0] / metric[1]), metric[1] / timer.stop()

def train_ch8(net, train_iter, vocab, lr, num_epochs, device,
              use_random_iter=False, use_plot=True, loss_fn=None, verbose=False):
    """训练 RNN
    Args:
        loss_fn: 可选，自定义损失函数（签名 loss(logits, labels) → 标量）
                 默认 nn.CrossEntropyLoss()
    """
    loss = loss_fn if loss_fn is not None else nn.CrossEntropyLoss()
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(
            net, train_iter, loss, updater, device, use_random_iter)
        if verbose and (epoch + 1) % 10 == 0:
            print(f"epoch {epoch + 1}: 困惑度 {ppl:.3f}")
    print(f"困惑度 {ppl:.1f}, {speed:.1f} 词元/秒 {str(device)}")
    print(predict("time traveller"))
    print(predict("traveller"))

---
### 9.1.7.1.PyPTO 训练说明

训练流程与 [8.5 节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)基本一致，但 GRU 模型有三个特殊之处：
1. **隐状态初始化**：每批数据需调用 `net.begin_state()` 初始化隐状态
2. **标签展平**：GRU 输出 shape 为 `(batch*steps, vocab)`，标签需从 `(batch, steps)` 展平为 `(batch*steps,)`
3. **梯度裁剪**：更新参数前先裁剪梯度，防止长序列反向传播导致梯度爆炸

`loss.backward()` 时 PyTorch 沿计算图依次调用各 `autograd.Function.backward()`，触发对应的 pypto 反向 kernel。**首次运行**会触发编译（耗时较长），编译完成后直接复用缓存。


In [13]:
# 预热：触发所有 PyPTO kernel 的首次编译
print("正在编译 pypto kernel（首次运行耗时较长，请耐心等待）...")
# 参数已在 get_params 中按 device 创建于 NPU 上，此处仅确保一致性
for p in net.params:
    p.data = p.data.to(device)
# 取一个批次，构造完整前向+反向计算来触发 JIT 编译
X, y = next(iter(train_iter))
X, y = X.to(device), y.to(device)
state = net.begin_state(X.shape[0], device)  # GRU 需要初始隐状态
y_hat, state = net(X, state)
y = y.T.reshape(-1).to(device)               # 标签展平为 (batch*steps,)
l = loss_fn(y_hat, y.long(), num_classes=len(vocab))
l.backward()
# 重置梯度，为正式训练做准备
for p in net.params:
    if p.grad is not None:
        p.grad = None
print("编译完成（耗时较长）。接下来可以正常训练了。")

In [14]:
num_epochs, lr = 500, 1
train_ch8(net, train_iter, vocab, lr, num_epochs, device, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

困惑度 1.0, 4024.2 词元/秒 npu:0


time travelleryou can show black is white by argument said filby


travelleryou can show black is white by argument said filby


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epochs, lr = 500, 1
model = d2l.RNNModelScratch(len(vocab), num_hiddens, device, get_params,
                            init_gru_state, gru)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)
    </pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    """训练网络一个迭代周期（定义见第8章）"""
    state, timer = None, d2l.Timer()
    metric = d2l.Accumulator(2)
    for X, Y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state.detach_()
            else:
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric.add(l * y.numel(), y.numel())
    return math.exp(metric[0] / metric[1]), metric[1] / timer.stop()
    </pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def train_ch8(net, train_iter, vocab, lr, num_epochs, device,
              use_random_iter=False):
    """训练模型（定义见第8章）"""
    loss = nn.CrossEntropyLoss()
    animator = d2l.Animator(xlabel="epoch", ylabel="perplexity",
                            legend=["train"], xlim=[10, num_epochs])
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: d2l.sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(
            net, train_iter, loss, updater, device, use_random_iter)
        if (epoch + 1) % 10 == 0:
            print(predict("time traveller"))
            animator.add(epoch + 1, [ppl])
    print(f"困惑度 {ppl:.1f}, {speed:.1f} 词元/秒 {str(device)}")
    print(predict("time traveller"))
    print(predict("traveller"))
    </pre>
  </div>
</details>


GRU 从零实现训练 500 轮后**困惑度降至 1.0**，预测文本已相当通顺；训练速度约 4024 词元/秒。与 [8.5 节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb) 的普通 RNN 相比，GRU 通过重置门与更新门缓解了长序列信息遗忘问题。

---
## 9.1.8.简洁实现

高级API包含了前文介绍的所有配置细节， 所以我们可以直接实例化门控循环单元模型。 这段代码的运行速度要快得多， 因为它使用的是编译好的运算符而不是Python来处理之前阐述的许多细节。

In [15]:
from src.pypto_ops import PyPTOGRU, PyPTOLinear

class PyPTORNNModel(nn.Module):
    """基于 PyPTO 模块的 RNN 模型（简洁版）。"""
    def __init__(self, rnn_layer, vocab_size):
        super().__init__()
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = PyPTOLinear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            self.linear = PyPTOLinear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size).type(torch.float32)
        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape(-1, Y.shape[-1]))
        return output, state

    def begin_state(self, device, batch_size=1):
        return torch.zeros((self.num_directions * self.rnn.num_layers,
                            batch_size, self.num_hiddens), device=device)

num_inputs = len(vocab)
gru_layer = PyPTOGRU(num_inputs, num_hiddens)
model_concise = PyPTORNNModel(gru_layer, len(vocab))
model_concise = model_concise.to(device)
train_ch8(model_concise, train_iter, vocab, lr, num_epochs, device, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))


困惑度 1.1, 4852.8 词元/秒 npu:0


time traveller for so it will be convenient to speak of himwas e


travelleryou can show black is white by argument said filby


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_inputs = vocab_size
gru_layer = nn.GRU(num_inputs, num_hiddens)
model = d2l.RNNModel(gru_layer, len(vocab))
model = model.to(device)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)
    </pre>
  </div>
</details>


简洁实现（基于封装好的 `PyPTOGRU` 算子）**困惑度为 1.1**，与从零实现相当，速度约 4852 词元/秒，略快于从零实现——这就是算子封装带来的编译与执行收益。

---
## 9.1.9.小结

* 门控循环神经网络可以更好地捕获时间步距离很长的序列上的依赖关系。
* 重置门有助于捕获序列中的短期依赖关系。
* 更新门有助于捕获序列中的长期依赖关系。
* 重置门打开时，门控循环单元包含基本循环神经网络；更新门打开时，门控循环单元可以跳过子序列。

---
## 9.1.10.练习

1. 假设我们只想使用时间步$t'$的输入来预测时间步$t > t'$的输出。对于每个时间步，重置门和更新门的最佳值是什么？
1. 调整和分析超参数对运行时间、困惑度和输出顺序的影响。
1. 比较`rnn.RNN`和`rnn.GRU`的不同实现对运行时间、困惑度和输出字符串的影响。
1. 如果仅仅实现门控循环单元的一部分，例如，只有一个重置门或一个更新门会怎样？


参考答案详见 [answers/09.01_reference_answer](./answers/09.01_reference_answer.ipynb)。


### 9.1.10.1.参考答案（PyPTO）

In [16]:
!cat answers/txt/09.01_reference_answer_pypto.txt

### 9.1.10.2.参考答案（PyTorch）

In [17]:
!cat answers/txt/09.01_reference_answer_pytorch.txt